# Phase 37 — EDA + Targeted Augmentation

Data augmentation για βελτίωση των σπάνιων κλάσεων.

**EDA (Easy Data Augmentation) — Wei & Zou 2019:**
1. **SR (Synonym Replacement):** Αντικατάσταση λέξεων με συνώνυμα
2. **RI (Random Insertion):** Εισαγωγή συνώνυμων λέξεων
3. **RS (Random Swap):** Εναλλαγή δύο τυχαίων λέξεων
4. **RD (Random Deletion):** Διαγραφή τυχαίων λέξεων

**Targeted Augmentation:** EDA μόνο στις σπάνιες κλάσεις
(κλάσεις με λιγότερα από 50 δείγματα στο train+valid)

**Στρατηγική:**
- Εφαρμόζουμε σε train+valid
- Αξιολογούμε με cross-validation
- Εκπαιδεύουμε BERT-base Focal στο augmented dataset

In [ ]:
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('omw-1.4', quiet=True)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import random
import copy
from collections import Counter
from nltk.corpus import wordnet
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
train = pd.read_csv('/train.csv')
valid = pd.read_csv('/valid.csv')
test  = pd.read_csv('/test.csv')

# Train + Valid μαζί
train_full = pd.concat([train, valid], ignore_index=True)

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_full = make_text(train_full)
texts_test = make_text(test)

print(f'Train+Valid: {len(train_full)} | Test: {len(test)}')
print('\n=== ΚΑΤΑΝΟΜΗ HAZARD ΚΛΑΣΕΩΝ ===')
hazard_counts = train_full['hazard-category'].value_counts()
for cls, cnt in hazard_counts.items():
    flag = ' ← σπάνια' if cnt < 50 else ''
    print(f'  {cls:35s}: {cnt:4d}{flag}')

## EDA Functions

In [ ]:
# Domain-specific words που ΔΕΝ πρέπει να αντικατασταθούν
STOP_WORDS = {
    'salmonella', 'listeria', 'monocytogenes', 'allergen', 'allergens',
    'ethylene', 'oxide', 'pesticide', 'undeclared', 'biological',
    'chemical', 'foreign', 'bodies', 'migration', 'packaging',
    'recall', 'recalled', 'product', 'products', 'food', 'hazard',
    'contamination', 'contaminated', 'bacteria', 'pathogen',
    'salmonella', 'ecoli', 'coli', 'hepatitis', 'norovirus'
}

def get_synonyms(word):
    """Βρίσκει συνώνυμα από WordNet, εξαιρώντας domain-specific λέξεις."""
    if word.lower() in STOP_WORDS:
        return []
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace('_', ' ')
            if synonym.lower() != word.lower() and synonym.lower() not in STOP_WORDS:
                synonyms.add(synonym)
    return list(synonyms)


def synonym_replacement(words, n=1):
    """SR: Αντικατάσταση n λέξεων με συνώνυμα."""
    new_words = words.copy()
    replaceable = [i for i, w in enumerate(words)
                   if w.lower() not in STOP_WORDS and get_synonyms(w)]
    if not replaceable:
        return new_words
    for idx in random.sample(replaceable, min(n, len(replaceable))):
        synonyms = get_synonyms(words[idx])
        if synonyms:
            new_words[idx] = random.choice(synonyms)
    return new_words


def random_insertion(words, n=1):
    """RI: Εισαγωγή n συνώνυμων λέξεων σε τυχαία θέση."""
    new_words = words.copy()
    for _ in range(n):
        candidates = [w for w in words if w.lower() not in STOP_WORDS and get_synonyms(w)]
        if not candidates:
            break
        word = random.choice(candidates)
        synonyms = get_synonyms(word)
        if synonyms:
            insert_pos = random.randint(0, len(new_words))
            new_words.insert(insert_pos, random.choice(synonyms))
    return new_words


def random_swap(words, n=1):
    """RS: Εναλλαγή n ζευγών τυχαίων λέξεων."""
    new_words = words.copy()
    for _ in range(n):
        if len(new_words) < 2:
            break
        i, j = random.sample(range(len(new_words)), 2)
        new_words[i], new_words[j] = new_words[j], new_words[i]
    return new_words


def random_deletion(words, p=0.1):
    """RD: Διαγραφή λέξεων με πιθανότητα p (εκτός domain words)."""
    if len(words) == 1:
        return words
    return [w for w in words
            if w.lower() in STOP_WORDS or random.random() > p]


def eda(text, alpha_sr=0.1, alpha_ri=0.1, alpha_rs=0.1, alpha_rd=0.1, n_aug=1):
    """Εφαρμόζει EDA και επιστρέφει n_aug augmented versions."""
    words = text.split()
    n     = max(1, int(len(words) * alpha_sr))
    augmented = []

    for _ in range(n_aug):
        op = random.choice(['sr', 'ri', 'rs', 'rd'])
        if op == 'sr':
            new_words = synonym_replacement(words, n)
        elif op == 'ri':
            new_words = random_insertion(words, n)
        elif op == 'rs':
            new_words = random_swap(words, n)
        else:
            new_words = random_deletion(words, alpha_rd)
        augmented.append(' '.join(new_words))

    return augmented


# Test
sample = 'recalled product contains undeclared milk allergen causing reaction'
print('Original:', sample)
for aug in eda(sample, n_aug=3):
    print('Augmented:', aug)

## Targeted Augmentation — Σπάνιες Κλάσεις

In [ ]:
# Threshold για σπάνιες κλάσεις
RARE_THRESHOLD = 50
N_AUG_PER_SAMPLE = 3  # Κάθε σπάνιο δείγμα → 3 augmented versions

print(f'=== TARGETED AUGMENTATION (threshold={RARE_THRESHOLD}) ===\n')

augmented_texts   = list(texts_full)
augmented_hazard  = list(train_full['hazard-category'])
augmented_product = list(train_full['product-category'])

rare_classes = hazard_counts[hazard_counts < RARE_THRESHOLD].index.tolist()
print(f'Σπάνιες hazard κλάσεις: {rare_classes}\n')

for i, (text, hazard, product) in enumerate(zip(
        texts_full,
        train_full['hazard-category'],
        train_full['product-category'])):

    if hazard in rare_classes:
        aug_texts = eda(text, n_aug=N_AUG_PER_SAMPLE)
        augmented_texts.extend(aug_texts)
        augmented_hazard.extend([hazard] * N_AUG_PER_SAMPLE)
        augmented_product.extend([product] * N_AUG_PER_SAMPLE)

print(f'Πριν augmentation:  {len(texts_full)} δείγματα')
print(f'Μετά augmentation:  {len(augmented_texts)} δείγματα')
print(f'Νέα δείγματα:       {len(augmented_texts) - len(texts_full)}')

print('\n=== ΝΕΕΣ ΚΑΤΑΝΟΜΕΣ ΣΠΑΝΙΩΝ ΚΛΑΣΕΩΝ ===')
new_counts = Counter(augmented_hazard)
for cls in rare_classes:
    old = hazard_counts[cls]
    new = new_counts[cls]
    print(f'  {cls:35s}: {old} → {new} (+{new-old})')

## Εκπαίδευση BERT-base + Focal Loss στο Augmented Dataset

In [ ]:
# Label encoding
le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(augmented_hazard)
le_product.fit(augmented_product)

y_aug_hazard  = le_hazard.transform(augmented_hazard)
y_aug_product = le_product.transform(augmented_product)

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)
print(f'Augmented dataset: {len(augmented_texts)} δείγματα')
print(f'Hazard classes: {n_hazard} | Product classes: {n_product}')


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, reduction='none')
        pt   = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


MODEL_NAME   = 'bert-base-uncased'
BATCH_SIZE   = 32
MAX_LENGTH   = 128
LR           = 2e-5
N_EPOCHS     = 20
WARMUP_RATIO = 0.1

tokenizer    = AutoTokenizer.from_pretrained(MODEL_NAME)
criterion    = FocalLoss(gamma=2.0)
dummy_labels = np.zeros(len(texts_test), dtype=int)
print(f'Tokenizer loaded: {MODEL_NAME}')

In [ ]:
class FoodHazardDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in loader:
        labels = batch.pop('label').to(device)
        batch  = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        loss = criterion(model(**batch).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def get_probabilities(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            probs = torch.softmax(model(**batch).logits, dim=1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)


def official_st1_score(y_true_h, y_pred_h, y_true_p, y_pred_p, verbose=True):
    f1_h = f1_score(y_true_h, y_pred_h, average='macro', zero_division=0)
    mask = (np.array(y_true_h) == np.array(y_pred_h))
    f1_p = f1_score(
        np.array(y_true_p)[mask], np.array(y_pred_p)[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_h + f1_p) / 2
    if verbose:
        print(f'  macro-F1 Hazard:                    {f1_h:.4f}')
        print(f'  Σωστά hazard:                       {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'  macro-F1 Product (given correct h): {f1_p:.4f}')
        print(f'  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'  OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

In [ ]:
# DataLoaders
aug_loader_hazard  = DataLoader(FoodHazardDataset(augmented_texts, y_aug_hazard,  tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
aug_loader_product = DataLoader(FoodHazardDataset(augmented_texts, y_aug_product, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
test_loader_hazard  = DataLoader(FoodHazardDataset(texts_test, dummy_labels, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
test_loader_product = DataLoader(FoodHazardDataset(texts_test, dummy_labels, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

print(f'Augmented train batches: {len(aug_loader_hazard)}')
print(f'(vs original: ~{len(texts_full)//BATCH_SIZE} batches)')

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ BERT-base + Focal Loss (Augmented) — HAZARD ===')
print(f'Dataset: {len(augmented_texts)} δείγματα | Epochs: {N_EPOCHS}\n')

bert_hazard  = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_hazard).to(device)
opt_h        = AdamW(bert_hazard.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(aug_loader_hazard) * N_EPOCHS
sch_h        = get_linear_schedule_with_warmup(opt_h, int(total_steps*WARMUP_RATIO), total_steps)

for epoch in range(N_EPOCHS):
    loss = train_epoch(bert_hazard, aug_loader_hazard, opt_h, sch_h)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {loss:.4f}')

print('Hazard εκπαίδευση ολοκληρώθηκε')

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ BERT-base + Focal Loss (Augmented) — PRODUCT ===')
print(f'Dataset: {len(augmented_texts)} δείγματα | Epochs: {N_EPOCHS}\n')

bert_product = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_product).to(device)
opt_p        = AdamW(bert_product.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(aug_loader_product) * N_EPOCHS
sch_p        = get_linear_schedule_with_warmup(opt_p, int(total_steps*WARMUP_RATIO), total_steps)

for epoch in range(N_EPOCHS):
    loss = train_epoch(bert_product, aug_loader_product, opt_p, sch_p)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {loss:.4f}')

print('Product εκπαίδευση ολοκληρώθηκε')

In [ ]:
# Test predictions
test_hazard_probs  = get_probabilities(bert_hazard,  test_loader_hazard)
test_product_probs = get_probabilities(bert_product, test_loader_product)

np.save('aug_bert_test_hazard_probs.npy',  test_hazard_probs)
np.save('aug_bert_test_product_probs.npy', test_product_probs)

test_hazard  = le_hazard.inverse_transform(test_hazard_probs.argmax(axis=1))
test_product = le_product.inverse_transform(test_product_probs.argmax(axis=1))

pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_hazard,
    'product-category': test_product
}).to_csv('submission_aug_bert.csv', index=False)

print('Αποθηκεύτηκε: submission_aug_bert.csv')
print('\n=== ΣΥΓΚΡΙΣΗ ===')
print('BERT-base Focal (original): 0.80399')
print('Best ensemble:              0.81882')